# 16_E8 FIXED - Al-Kafri mask decode and GT source comparison

Este notebook continúa el rescate de Al-Kafri después de reconstruir el pairing oficial con `Slices Numbers.csv` leído con `header=None`.

Objetivo: mejorar la visualización/decodificación de máscaras comparando fuentes de ground truth y estrategias de decodificación.

No entrena modelos. Genera figuras diagnósticas y CSVs para decidir si se puede crear un subset axial entrenable.


In [ ]:
import importlib.util, subprocess, sys

def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name])

ensure_package('pydicom')
ensure_package('skimage', 'scikit-image')

import ast, json, re, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom
from PIL import Image
from scipy import ndimage
from skimage.filters import threshold_multiotsu
from skimage.morphology import remove_small_objects, binary_closing, disk
from skimage.transform import resize
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 160)
pd.set_option('display.max_colwidth', 180)

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as exc:
    print('Drive no montado automáticamente:', repr(exc))


In [ ]:
PFI_ROOT = Path('/content/drive/MyDrive/PFI_MVP')
ALKAFRI_ROOT = PFI_ROOT / 'data' / 'AXIAL_ALKAFRI'
EXTRACTED_ROOT = ALKAFRI_ROOT / 'extracted'
NESTED_ROOT = EXTRACTED_ROOT / '_nested'
E7_ROOT = PFI_ROOT / 'results' / 'E7_alkafri_axial_curated_subset'
E8_ROOT = PFI_ROOT / 'results' / 'E8_alkafri_official_pairing'
FIGURES_ROOT = PFI_ROOT / 'figures'
DOCS_ROOT = PFI_ROOT / 'docs'

for p in [E8_ROOT, FIGURES_ROOT, DOCS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

OFFICIAL_FILTERED_PATH = E8_ROOT / 'E8_official_pairing_candidates_HEADER_NONE_FILTERED.csv'
GT_INDEX_PATH = E7_ROOT / 'E7_alkafri_gt_case_index.csv'

print('PFI_ROOT:', PFI_ROOT)
print('OFFICIAL_FILTERED_PATH:', OFFICIAL_FILTERED_PATH, '| exists:', OFFICIAL_FILTERED_PATH.exists())
print('GT_INDEX_PATH:', GT_INDEX_PATH, '| exists:', GT_INDEX_PATH.exists())

if not OFFICIAL_FILTERED_PATH.exists():
    raise FileNotFoundError('Falta E8_official_pairing_candidates_HEADER_NONE_FILTERED.csv. Ejecutar primero el patch de pairing oficial HEADER_NONE.')
if not GT_INDEX_PATH.exists():
    raise FileNotFoundError('Falta E7_alkafri_gt_case_index.csv. Ejecutar primero notebooks E7/E8 base.')


In [ ]:
def norm_case(x):
    if pd.isna(x):
        return None
    m = re.search(r'(\d{1,4})', str(x))
    return f'{int(m.group(1)):04d}' if m else None

def infer_gt_type(path):
    s = str(path).lower()
    if '05_final' in s or 'final_ground_truth' in s:
        return 'final'
    if '04_intermediary' in s or 'intermediary' in s:
        return 'intermediary'
    if '03_manual' in s or 'manual_label' in s or 'labeller' in s:
        return 'manual'
    return 'unknown'

def parse_gt_filename(path):
    name = Path(path).name
    m = re.search(r'(T[12])[_-](\d{1,4})[_-]D([345])', name, re.I)
    if not m:
        return None, None, None
    return m.group(1).upper(), f'{int(m.group(2)):04d}', int(m.group(3))

def read_dicom(path):
    ds = pydicom.dcmread(str(path), force=True)
    _ = ds.pixel_array
    return ds

def normalize_image(arr):
    arr = arr.astype(np.float32)
    p1, p99 = np.percentile(arr, [1, 99])
    if p99 <= p1:
        return np.zeros_like(arr, dtype=np.float32)
    return (np.clip(arr, p1, p99) - p1) / (p99 - p1 + 1e-8)

def read_mask_raw(path):
    with Image.open(path) as img:
        arr = np.asarray(img)
    if arr.ndim == 3 and arr.shape[-1] == 4:
        arr = arr[..., :3]
    return arr

def to_gray_for_decode(arr):
    if arr.ndim == 2:
        return arr.astype(np.float32)
    return arr[..., :3].astype(np.float32).mean(axis=-1)

def border_values(arr):
    a = to_gray_for_decode(arr)
    return np.concatenate([a[0, :], a[-1, :], a[:, 0], a[:, -1]])

def dominant_scalar(values):
    vals = np.asarray(values).reshape(-1)
    if vals.size == 0:
        return None
    vals_i = np.round(vals).astype(np.int32)
    c = Counter(vals_i.tolist())
    return c.most_common(1)[0][0]

def clean_binary(mask, min_size=80, close_radius=1):
    mask = mask.astype(bool)
    if close_radius and mask.any():
        mask = binary_closing(mask, disk(close_radius))
    if mask.any():
        mask = remove_small_objects(mask, min_size=min_size)
    return mask.astype(np.uint8)

def component_count(mask):
    _, n = ndimage.label(mask.astype(bool))
    return int(n)

def resize_mask_if_needed(mask, target_shape):
    if tuple(mask.shape) == tuple(target_shape):
        return mask.astype(np.uint8)
    return (resize(mask.astype(np.float32), target_shape, order=0, preserve_range=True, anti_aliasing=False) > 0.5).astype(np.uint8)

def unique_profile(arr, max_items=15):
    if arr.ndim == 3:
        flat = arr.reshape(-1, arr.shape[-1])
        vals, counts = np.unique(flat, axis=0, return_counts=True)
        values = [tuple(map(int, v.tolist())) for v in vals]
    else:
        vals, counts = np.unique(arr.reshape(-1), return_counts=True)
        values = [int(v) for v in vals]
    order = np.argsort(counts)[::-1]
    total = int(np.prod(arr.shape[:2]))
    rows = []
    for idx in order[:max_items]:
        rows.append({'value': str(values[idx]), 'count': int(counts[idx]), 'ratio': float(counts[idx] / total)})
    return rows, len(vals)


In [ ]:
# 1) Cargar candidatos oficiales filtrados y reconstruir pool de GT por fuente
official_df = pd.read_csv(OFFICIAL_FILTERED_PATH)
official_df['case_id_norm'] = official_df['case_id'].apply(norm_case)
official_df['disc_id'] = pd.to_numeric(official_df['disc_id'], errors='coerce').astype('Int64')
official_df['slice_offset'] = pd.to_numeric(official_df['slice_offset'], errors='coerce')

gt_df = pd.read_csv(GT_INDEX_PATH, low_memory=False)
gt_df = gt_df[gt_df['extension'].astype(str).str.lower().eq('.png')].copy()
gt_df['gt_type_fixed'] = gt_df['gt_file_path'].apply(infer_gt_type)
parsed = gt_df['gt_file_path'].apply(parse_gt_filename)
gt_df['modality_from_name'] = parsed.apply(lambda x: x[0])
gt_df['case_id_norm_from_name'] = parsed.apply(lambda x: x[1])
gt_df['disc_id_from_name'] = parsed.apply(lambda x: x[2])
gt_df['case_id_norm'] = gt_df['case_id_norm_from_name'].fillna(gt_df['case_id'].apply(norm_case))
gt_df['modality'] = gt_df['modality_from_name'].fillna(gt_df.get('modality'))
gt_df['disc_id'] = pd.to_numeric(gt_df['disc_id_from_name'].fillna(gt_df.get('disc_or_slice_id')), errors='coerce').astype('Int64')
gt_df['gt_type'] = gt_df['gt_type_fixed'].where(gt_df['gt_type_fixed'].ne('unknown'), gt_df.get('gt_type'))

gt_df = gt_df[gt_df['case_id_norm'].notna() & gt_df['modality'].notna() & gt_df['disc_id'].notna()].copy()
gt_df['gt_priority'] = gt_df['gt_type'].map({'final': 0, 'intermediary': 1, 'manual': 2}).fillna(9).astype(int)

print('official_df:', official_df.shape)
print('gt_df png parsed:', gt_df.shape)
display(gt_df.groupby(['gt_type', 'modality'], dropna=False).size().reset_index(name='n').sort_values(['gt_type', 'modality']))

# Crear una tabla comparativa de fuentes disponibles por caso/modalidad/disco
gt_source_summary = (
    gt_df.groupby(['case_id_norm', 'modality', 'disc_id', 'gt_type'], dropna=False)
    .agg(n_files=('gt_file_path', 'count'), first_path=('gt_file_path', 'first'))
    .reset_index()
)
gt_source_summary.to_csv(E8_ROOT / 'E8_FIXED_gt_sources_by_case_modality_disc.csv', index=False)
display(gt_source_summary.head(30))

# Seleccionar mejor GT disponible para cada candidato oficial: final > intermediary > manual
base_cols = [c for c in official_df.columns if c not in ['gt_file_path', 'gt_type']]
merged = official_df[base_cols].merge(
    gt_df[['case_id_norm', 'modality', 'disc_id', 'gt_file_path', 'gt_type', 'gt_priority']],
    on=['case_id_norm', 'modality', 'disc_id'],
    how='left'
)
merged = merged.sort_values(['case_id_norm', 'modality', 'disc_id', 'image_file_path', 'gt_priority', 'gt_file_path'])
selected_df = merged.drop_duplicates(subset=['case_id_norm', 'modality', 'disc_id', 'image_file_path'], keep='first').copy()
selected_df = selected_df[selected_df['gt_file_path'].notna()].copy()

selected_df.to_csv(E8_ROOT / 'E8_FIXED_official_candidates_best_gt_source.csv', index=False)
print('selected_df:', selected_df.shape)
display(selected_df.groupby(['gt_type', 'modality'], dropna=False).size().reset_index(name='n').sort_values(['gt_type', 'modality']))
display(selected_df.head(20))


In [ ]:
# 2) Estrategias de decodificación mejoradas

def decode_exact_label_like(raw, min_ratio=0.0005, max_ratio=0.40, min_size=80):
    # Sirve cuando la máscara es realmente label map discreto.
    gray = to_gray_for_decode(raw)
    gi = np.round(gray).astype(np.int32)
    vals, counts = np.unique(gi.reshape(-1), return_counts=True)
    total = gi.size
    border_bg = dominant_scalar(border_values(gi))
    mask = np.zeros_like(gi, dtype=bool)
    used = []
    for v, c in zip(vals, counts):
        ratio = c / total
        if int(v) == int(border_bg):
            continue
        if min_ratio <= ratio <= max_ratio:
            mask |= (gi == v)
            used.append(int(v))
    return clean_binary(mask, min_size=min_size, close_radius=1), {'method': 'exact_label_like', 'used_values': used, 'unique_count': int(len(vals)), 'border_bg': int(border_bg)}

def decode_range_clean(raw, top_k=8, tolerance=2, min_ratio=0.004, max_ratio=0.18, min_size=120):
    # Agrupa valores cercanos para evitar speckles por variaciones pequeñas de intensidad.
    gray = to_gray_for_decode(raw)
    gi = np.round(gray).astype(np.int32)
    vals, counts = np.unique(gi.reshape(-1), return_counts=True)
    total = gi.size
    border_bg = dominant_scalar(border_values(gi))
    order = np.argsort(counts)[::-1]
    mask = np.zeros_like(gi, dtype=bool)
    used = []
    for idx in order:
        v = int(vals[idx])
        ratio = counts[idx] / total
        if v == border_bg:
            continue
        if min_ratio <= ratio <= max_ratio:
            mask |= np.abs(gi - v) <= tolerance
            used.append(v)
        if len(used) >= top_k:
            break
    return clean_binary(mask, min_size=min_size, close_radius=2), {'method': 'range_clean_pm2', 'used_values': used, 'unique_count': int(len(vals)), 'border_bg': int(border_bg)}

def decode_multiotsu_non_border(raw, classes=5, min_size=200):
    # Para GT intermedio tipo mapa continuo: segmenta en bandas y excluye clase dominante del borde.
    gray = to_gray_for_decode(raw)
    if np.nanmax(gray) <= np.nanmin(gray):
        return np.zeros_like(gray, dtype=np.uint8), {'method': 'multiotsu_non_border', 'classes': 0, 'border_class': None}
    # multiotsu falla si hay muy pocos niveles; se reduce classes si hace falta.
    unique_count = len(np.unique(np.round(gray).astype(np.int32)))
    c = min(classes, max(2, unique_count))
    try:
        th = threshold_multiotsu(gray, classes=c)
        regions = np.digitize(gray, th)
    except Exception:
        q = np.percentile(gray, [20, 40, 60, 80])
        regions = np.digitize(gray, q)
        th = q
    border_class = dominant_scalar(border_values(regions))
    mask = regions != border_class
    mask = clean_binary(mask, min_size=min_size, close_radius=2)
    return mask, {'method': 'multiotsu_non_border', 'thresholds': [float(x) for x in np.asarray(th).reshape(-1)], 'border_class': int(border_class), 'unique_count': int(unique_count)}

def select_best_decode(raw):
    # Genera varias versiones. La selección automática es solo diagnóstica; la decisión final es visual.
    raw_profile, unique_count = unique_profile(raw, max_items=12)
    exact, info_exact = decode_exact_label_like(raw)
    range_m, info_range = decode_range_clean(raw)
    otsu_m, info_otsu = decode_multiotsu_non_border(raw)

    candidates = [
        ('exact_label_like', exact, info_exact),
        ('range_clean_pm2', range_m, info_range),
        ('multiotsu_non_border', otsu_m, info_otsu),
    ]

    scored = []
    for name, m, info in candidates:
        ratio = float(m.mean())
        comps = component_count(m)
        # Penaliza máscaras vacías, enormes o con demasiados componentes.
        score = 0
        if 0.005 <= ratio <= 0.45:
            score += 2
        if comps <= 80:
            score += 2
        elif comps <= 250:
            score += 1
        if ratio < 0.002 or ratio > 0.65:
            score -= 3
        scored.append((score, name, m, {**info, 'foreground_ratio': ratio, 'components': comps, 'raw_profile': raw_profile}))

    scored = sorted(scored, key=lambda x: x[0], reverse=True)
    return scored[0][2], {'selected_method': scored[0][1], 'score': int(scored[0][0]), **scored[0][3]}, {name: (m, info) for _, name, m, info in scored}

print('Funciones de decode listas.')


In [ ]:
# 3) Generar figuras FIXED comparando raw GT y 3 decodificaciones

VIS_N = 120

vis_df = (
    selected_df
    .sort_values(['case_id_norm', 'modality', 'disc_id', 'instance_number'])
    .groupby(['modality', 'disc_id'], group_keys=False)
    .head(25)
    .head(VIS_N)
    .copy()
)

print('Candidatos FIXED a visualizar:', len(vis_df))
review_rows = []

for i, (_, r) in enumerate(tqdm(vis_df.iterrows(), total=len(vis_df), desc='fixed mask decode'), start=1):
    cid = f'fixed_mask_decode_{i:03d}'
    try:
        ds = read_dicom(r['image_file_path'])
        img = ds.pixel_array.astype(np.float32)
        img_norm = normalize_image(img)
        raw = read_mask_raw(r['gt_file_path'])

        best_mask, best_info, all_decodes = select_best_decode(raw)
        best_mask = resize_mask_if_needed(best_mask, img.shape)

        exact_m, exact_i = all_decodes.get('exact_label_like')
        range_m, range_i = all_decodes.get('range_clean_pm2')
        otsu_m, otsu_i = all_decodes.get('multiotsu_non_border')
        exact_m = resize_mask_if_needed(exact_m, img.shape)
        range_m = resize_mask_if_needed(range_m, img.shape)
        otsu_m = resize_mask_if_needed(otsu_m, img.shape)

        raw_profile, unique_count = unique_profile(raw, max_items=10)

        fig, axes = plt.subplots(1, 6, figsize=(24, 4))
        axes[0].imshow(img_norm, cmap='gray')
        axes[0].set_title('Axial IMA')
        axes[0].axis('off')

        axes[1].imshow(raw if raw.ndim == 3 else raw, cmap=None if raw.ndim == 3 else 'nipy_spectral')
        axes[1].set_title(f'Raw GT\n{r["gt_type"]} | unique={unique_count}')
        axes[1].axis('off')

        for ax, m, title in [
            (axes[2], exact_m, f'exact label-like\nratio={exact_m.mean():.3f} comps={component_count(exact_m)}'),
            (axes[3], range_m, f'range clean ±2\nratio={range_m.mean():.3f} comps={component_count(range_m)}'),
            (axes[4], otsu_m, f'multiotsu non-border\nratio={otsu_m.mean():.3f} comps={component_count(otsu_m)}'),
            (axes[5], best_mask, f'BEST: {best_info["selected_method"]}\nratio={best_mask.mean():.3f} comps={component_count(best_mask)}'),
        ]:
            ax.imshow(img_norm, cmap='gray')
            ax.imshow(np.ma.masked_where(m <= 0, m), alpha=0.45, cmap='autumn')
            ax.set_title(title)
            ax.axis('off')

        fig.suptitle(
            f'{cid} | official HEADER_NONE | case={r["case_id_norm"]} | {r["modality"]} | D={r["disc_id"]} | '
            f'slice={r.get("slice_number", "?")} | inst={r.get("instance_number", "?")} | gt={r["gt_type"]}',
            fontsize=10
        )
        fig.tight_layout()
        fig_path = FIGURES_ROOT / f'E8_alkafri_FIXED_MASK_DECODE_{i:03d}.png'
        fig.savefig(fig_path, dpi=150, bbox_inches='tight')
        plt.close(fig)

        review_rows.append({
            'candidate_id': cid,
            'figure_path': str(fig_path),
            'strategy': r.get('strategy'),
            'case_id': r['case_id_norm'],
            'modality': r['modality'],
            'disc_id': int(r['disc_id']),
            'slice_number': r.get('slice_number'),
            'instance_number': r.get('instance_number'),
            'gt_type': r['gt_type'],
            'gt_file_path': r['gt_file_path'],
            'image_file_path': r['image_file_path'],
            'raw_unique_count': int(unique_count),
            'raw_profile': json.dumps(raw_profile, ensure_ascii=False),
            'selected_decode_method': best_info['selected_method'],
            'selected_foreground_ratio': float(best_mask.mean()),
            'selected_component_count': component_count(best_mask),
            'decode_info': json.dumps(best_info, ensure_ascii=False),
            'auto_status': 'review',
            'manual_accept': '',
            'manual_reject_reason': '',
            'notes': '',
        })

    except Exception as exc:
        review_rows.append({
            'candidate_id': cid,
            'auto_status': 'error',
            'error': repr(exc),
            'case_id': r.get('case_id_norm'),
            'modality': r.get('modality'),
            'disc_id': r.get('disc_id'),
            'gt_file_path': r.get('gt_file_path'),
            'image_file_path': r.get('image_file_path'),
        })

review_fixed_df = pd.DataFrame(review_rows)
review_fixed_path = E8_ROOT / 'E8_FIXED_mask_decode_visual_review_sheet.csv'
review_fixed_df.to_csv(review_fixed_path, index=False)

print('Figuras FIXED generadas:', int(review_fixed_df.get('figure_path', pd.Series(dtype=object)).notna().sum()))
print('Review sheet:', review_fixed_path)
display(review_fixed_df.head(30))
display(review_fixed_df.groupby(['gt_type', 'modality', 'selected_decode_method'], dropna=False).size().reset_index(name='n').sort_values('n', ascending=False))


In [ ]:
# 4) Reporte final FIXED
report = {
    'notebook': '16_E8_alkafri_mask_decode_FIXED',
    'goal': 'compare final/intermediary/manual GT source and improve mask decoding visual diagnostics',
    'input_official_candidates': str(OFFICIAL_FILTERED_PATH),
    'official_candidates_loaded': int(len(official_df)),
    'gt_png_parsed': int(len(gt_df)),
    'selected_candidates_best_gt_source': int(len(selected_df)),
    'selected_by_gt_type_modality': selected_df.groupby(['gt_type', 'modality'], dropna=False).size().reset_index(name='n').to_dict(orient='records'),
    'visual_review_figures': int(review_fixed_df.get('figure_path', pd.Series(dtype=object)).notna().sum()),
    'review_sheet': str(E8_ROOT / 'E8_FIXED_mask_decode_visual_review_sheet.csv'),
    'figures_prefix': str(FIGURES_ROOT / 'E8_alkafri_FIXED_MASK_DECODE_###.png'),
    'decision': 'pending_visual_review',
    'recommendation': 'Abrir figuras FIXED. Si las máscaras limpias caen sobre regiones anatómicas coherentes en al menos 30 casos, crear subset axial curado. Si siguen siendo ruido/dispersión, el pairing está resuelto pero la decodificación de GT no es entrenable todavía.'
}

report_path = E8_ROOT / 'E8_FIXED_mask_decode_report.json'
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')

md = [
    '# E8 FIXED mask decode conclusion',
    '',
    f'- Official candidates loaded: {len(official_df)}',
    f'- Parsed GT PNG files: {len(gt_df)}',
    f'- Selected candidates with best GT source: {len(selected_df)}',
    f'- Fixed visual review figures: {report["visual_review_figures"]}',
    '',
    '## Interpretation',
    'The official pairing stage is considered recovered. This notebook evaluates whether the available GT sources can be decoded into trainable anatomical masks.',
    '',
    '## Next decision',
    report['recommendation'],
]
(DOCS_ROOT / 'E8_FIXED_mask_decode_conclusion.md').write_text('\n'.join(md), encoding='utf-8')

print(json.dumps(report, indent=2, ensure_ascii=False))
print('Reporte:', report_path)
